# 11 - Figure Generation

Generate all publication-quality figures for the paper.
All figures are saved at 300 DPI to `figures/`.

**Figures generated in this notebook**:
1. `fig_comparison.png` - TCGA vs GSE96058 model comparison (grouped bar)
2. `fig_roc.png` - Two-panel ROC curves
3. `fig_ablation.png` - Two-panel ablation study

**Figures generated in other notebooks** (already saved):
- `fig_kaplan_meier.png` (notebook 06)
- `fig_shap_beeswarm.png` (notebook 08)
- `fig_shap_pathway_only.png` (notebook 08)
- `fig_waterfall_high.png` (notebook 08)
- `fig_waterfall_low.png` (notebook 08)
- `fig_stability.png` (notebook 09)
- `fig_calibration.png` (notebook 10)

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.base import clone

from src.models import get_classifiers

# Publication styling
plt.rcParams.update({
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'legend.fontsize': 10,
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
})

## 1. Load Results Data

In [ ]:
performance = pd.read_csv('../results/03_model_performance.csv')
ablation = pd.read_csv('../results/ablation_results.csv')

print("Model Performance:")
print(performance.to_string(index=False))
print("\nAblation Results:")
print(ablation.to_string(index=False))

## 2. Figure: Model Comparison (TCGA vs GSE96058)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

models = ['Elastic Net', 'Random Forest', 'Gradient Boosting']
x = np.arange(len(models))
width = 0.35

tcga_aucs = [performance[(performance['Dataset'] == 'TCGA') & (performance['Model'] == m)]['AUC'].values[0] for m in models]
gse_data = performance[performance['Dataset'] == 'GSE96058']
# Map model names if they differ
gse_aucs = []
for m in models:
    match = gse_data[gse_data['Model'] == m]
    if len(match) == 0:
        match = gse_data[gse_data['Model'].str.contains(m.split()[0])]
    gse_aucs.append(match['AUC'].values[0] if len(match) > 0 else 0)

bars1 = ax.bar(x - width/2, tcga_aucs, width, label='TCGA (n=213)', color='#4472C4', edgecolor='white')
bars2 = ax.bar(x + width/2, gse_aucs, width, label='GSE96058 (n=1,483)', color='#C44E52', edgecolor='white')

# Add value labels
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

ax.axhline(y=0.75, color='gray', linestyle='--', alpha=0.5, label='AUC = 0.75')
ax.set_xlabel('Model')
ax.set_ylabel('AUC-ROC')
ax.set_title('Model Performance: TCGA (Training) vs GSE96058 (Validation)')
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.legend()
ax.set_ylim(0, 1.05)

plt.savefig('../figures/fig_comparison.png')
print("Saved figures/fig_comparison.png")
plt.show()

## 3. Figure: ROC Curves (Two-Panel)

In [ ]:
# Load TCGA data for ROC
from src.data_loader import load_tcga_feature_matrix

tcga = load_tcga_feature_matrix('../data/processed/02_tcga_feature_matrix.csv')
feature_cols = [c for c in tcga.columns if c.startswith('Pathway_') or c.startswith('Ratio_')]
X_tcga = tcga[feature_cols]
y_tcga = tcga['high_risk']

X_train, X_test, y_train, y_test = train_test_split(
    X_tcga, y_tcga, test_size=0.2, stratify=y_tcga, random_state=42
)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# Train Elastic Net on TCGA
en = get_classifiers()['Elastic Net']
en.fit(X_train_s, y_train)
en_proba_tcga = en.predict_proba(X_test_s)[:, 1]
fpr_tcga, tpr_tcga, _ = roc_curve(y_test, en_proba_tcga)
auc_tcga = roc_auc_score(y_test, en_proba_tcga)

In [ ]:
# For GSE96058 ROC, use out-of-fold predictions from GB on combined features
# (If expression data is available)
import os

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: TCGA Elastic Net
ax1 = axes[0]
ax1.plot(fpr_tcga, tpr_tcga, color='#4472C4', linewidth=2,
         label=f'Elastic Net (AUC = {auc_tcga:.3f})')
ax1.plot([0, 1], [0, 1], 'k--', alpha=0.5)
ax1.set_xlabel('False Positive Rate')
ax1.set_ylabel('True Positive Rate')
ax1.set_title('(A) TCGA - Elastic Net')
ax1.legend(loc='lower right')
ax1.set_aspect('equal')

# Right: GSE96058 Gradient Boosting (placeholder if no expression data)
ax2 = axes[1]
gse_exp_path = '../data/raw/GSE96058_gene_expression.csv'
if os.path.exists(gse_exp_path):
    from src.data_loader import load_clinical_data, load_gse96058_expression
    from src.preprocessing import zscore_normalize, encode_clinical_features, filter_outcome
    from src.features import compute_pathway_scores, add_ratio_features, build_feature_matrix
    
    gse_clin = load_clinical_data('../data/clinical/01_gse96058_clinical.csv')
    gse_clin = filter_outcome(gse_clin)
    gse_exp = load_gse96058_expression(gse_exp_path)
    
    common = list(set(gse_clin['sample_id']) & set(gse_exp['sample_id']))
    gse_clin = gse_clin[gse_clin['sample_id'].isin(common)].sort_values('sample_id').reset_index(drop=True)
    gse_exp = gse_exp[gse_exp['sample_id'].isin(common)].sort_values('sample_id').reset_index(drop=True)
    
    gse_exp_norm = zscore_normalize(gse_exp)
    pw = compute_pathway_scores(gse_exp_norm)
    pw = add_ratio_features(pw)
    cf = encode_clinical_features(gse_clin)
    X_gse = build_feature_matrix(pw, cf)
    y_gse = gse_clin['high_risk'].values
    
    # Collect out-of-fold ROC for GB
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    oof_proba = np.zeros(len(y_gse))
    gb_template = get_classifiers()['Gradient Boosting']
    
    for train_idx, test_idx in skf.split(np.array(X_gse), y_gse):
        sc = StandardScaler()
        Xtr = sc.fit_transform(np.array(X_gse)[train_idx])
        Xte = sc.transform(np.array(X_gse)[test_idx])
        gb = clone(gb_template)
        gb.fit(Xtr, y_gse[train_idx])
        oof_proba[test_idx] = gb.predict_proba(Xte)[:, 1]
    
    fpr_gse, tpr_gse, _ = roc_curve(y_gse, oof_proba)
    auc_gse = roc_auc_score(y_gse, oof_proba)
    
    ax2.plot(fpr_gse, tpr_gse, color='#C44E52', linewidth=2,
             label=f'Gradient Boosting (AUC = {auc_gse:.3f})')
else:
    ax2.text(0.5, 0.5, 'GSE96058 expression data\nnot available',
             ha='center', va='center', transform=ax2.transAxes, fontsize=12)

ax2.plot([0, 1], [0, 1], 'k--', alpha=0.5)
ax2.set_xlabel('False Positive Rate')
ax2.set_ylabel('True Positive Rate')
ax2.set_title('(B) GSE96058 - Gradient Boosting')
ax2.legend(loc='lower right')
ax2.set_aspect('equal')

plt.tight_layout()
plt.savefig('../figures/fig_roc.png')
print("Saved figures/fig_roc.png")
plt.show()

## 4. Figure: Ablation Study (Two-Panel)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# (A) Grouped bar chart
ax1 = axes[0]
feature_sets = ablation['Feature Set'].unique()
models = ablation['Model'].unique()
x = np.arange(len(feature_sets))
width = 0.25
colors = ['#4472C4', '#ED7D31', '#70AD47']

for i, model in enumerate(models):
    subset = ablation[ablation['Model'] == model]
    aucs = [subset[subset['Feature Set'] == fs]['AUC'].values[0] for fs in feature_sets]
    sds = [subset[subset['Feature Set'] == fs]['AUC_SD'].values[0] for fs in feature_sets]
    ax1.bar(x + i * width, aucs, width, yerr=sds, label=model,
            color=colors[i], edgecolor='white', capsize=3)

ax1.set_xlabel('Feature Set')
ax1.set_ylabel('AUC-ROC')
ax1.set_title('(A) Ablation Study: AUC by Feature Set')
ax1.set_xticks(x + width)
# Shorten labels
short_labels = ['Pathway\n(8)', 'Clinical\n(9)', 'Combined\n(17)']
ax1.set_xticklabels(short_labels)
ax1.legend()
ax1.set_ylim(0.5, 1.0)

# (B) ROC curves for Elastic Net across 3 feature sets
ax2 = axes[1]
if os.path.exists(gse_exp_path):
    # Generate ROC for each feature set using EN
    en_template = get_classifiers()['Elastic Net']
    
    feature_configs = {
        'Pathway Only': pw,
        'Clinical Only': cf,
        'Combined': X_gse,
    }
    roc_colors = ['#4472C4', '#ED7D31', '#70AD47']
    
    for (fs_name, X_fs), color in zip(feature_configs.items(), roc_colors):
        oof = np.zeros(len(y_gse))
        for train_idx, test_idx in skf.split(np.array(X_fs), y_gse):
            sc = StandardScaler()
            Xtr = sc.fit_transform(np.array(X_fs)[train_idx])
            Xte = sc.transform(np.array(X_fs)[test_idx])
            clf = clone(en_template)
            clf.fit(Xtr, y_gse[train_idx])
            oof[test_idx] = clf.predict_proba(Xte)[:, 1]
        
        fpr, tpr, _ = roc_curve(y_gse, oof)
        auc_val = roc_auc_score(y_gse, oof)
        ax2.plot(fpr, tpr, color=color, linewidth=2,
                 label=f'{fs_name} (AUC = {auc_val:.3f})')
else:
    ax2.text(0.5, 0.5, 'GSE96058 expression data\nnot available',
             ha='center', va='center', transform=ax2.transAxes, fontsize=12)

ax2.plot([0, 1], [0, 1], 'k--', alpha=0.5)
ax2.set_xlabel('False Positive Rate')
ax2.set_ylabel('True Positive Rate')
ax2.set_title('(B) Elastic Net ROC by Feature Set')
ax2.legend(loc='lower right')
ax2.set_aspect('equal')

plt.tight_layout()
plt.savefig('../figures/fig_ablation.png')
print("Saved figures/fig_ablation.png")
plt.show()

## 5. Verify All Figures Exist

In [ ]:
import os

expected_figures = [
    'fig_ablation.png',
    'fig_calibration.png',
    'fig_comparison.png',
    'fig_kaplan_meier.png',
    'fig_roc.png',
    'fig_shap_beeswarm.png',
    'fig_shap_pathway_only.png',
    'fig_stability.png',
    'fig_waterfall_high.png',
    'fig_waterfall_low.png',
]

print("Figure verification:")
for fig_name in expected_figures:
    path = f'../figures/{fig_name}'
    exists = os.path.exists(path)
    status = 'OK' if exists else 'MISSING'
    size = f'{os.path.getsize(path) / 1024:.0f} KB' if exists else '-'
    print(f"  [{status}] {fig_name:35s} {size}")